# 03 — Batch Screening Pipeline: Screen -> Extract -> Synthesize

Companion notebook to `../00-README.md` and `../04-deployment-architecture-aws-sagemaker.md`.

This notebook implements the shape of the batch screening pipeline end to end, on synthetic data:

1. **Screen** — a mocked, deterministic screening function applies inclusion/exclusion criteria to a
   synthetic corpus of abstracts (standing in for the self-hosted DeepSeek screening calls in
   production).
2. **Extract** — for abstracts that pass screening, pull structured data out (standing in for the
   extraction stage).
3. **Synthesize** — aggregate the extracted data into a summary, the way a draft synthesis would
   summarize the included evidence base.

A PRISMA-style funnel count (total -> excluded, by reason -> included -> extracted) is printed along
the way, mirroring the kind of accounting a real systematic review has to report.

Fully offline: standard library only, no model calls, no API keys.

## 1. A synthetic literature corpus

Twenty synthetic abstracts, generated with varied study type, indication, publication year, and
sample size — deliberately mixed so some abstracts pass every inclusion criterion and others fail on
exactly one, the way a real corpus does.

In [1]:
import random
import re

random.seed(3)

INDICATIONS = ["oncology", "cardiology"]
STUDY_TYPES = [
    "a randomized controlled trial",
    "a case report",
    "an observational cohort study",
    "a randomized controlled trial",   # weighted toward RCTs, like a real corpus skews
]

def build_corpus(n: int):
    corpus = []
    for i in range(n):
        indication = random.choice(INDICATIONS)
        study_type = random.choice(STUDY_TYPES)
        year = random.randint(2008, 2023)
        sample_size = random.randint(15, 600)
        text = (
            f"This report describes {study_type} evaluating an investigational agent in "
            f"patients with {indication} disease, published in {year}, enrolling {sample_size} patients."
        )
        corpus.append({"abstract_id": f"A{i:03d}", "text": text, "year": year})
    return corpus


corpus = build_corpus(20)
print(f"Synthetic corpus size: {len(corpus)} abstracts\n")
for a in corpus[:4]:
    print(a["abstract_id"], "-", a["text"])
print("...")


Synthetic corpus size: 20 abstracts

A000 - This report describes a case report evaluating an investigational agent in patients with oncology disease, published in 2019, enrolling 500 patients.
A001 - This report describes a randomized controlled trial evaluating an investigational agent in patients with oncology disease, published in 2023, enrolling 280 patients.
A002 - This report describes a case report evaluating an investigational agent in patients with oncology disease, published in 2023, enrolling 568 patients.
A003 - This report describes a randomized controlled trial evaluating an investigational agent in patients with cardiology disease, published in 2012, enrolling 252 patients.
...


## 2. Screening: apply inclusion/exclusion criteria

The screening function is deterministic and offline here (standing in for a real screening call to
the self-hosted DeepSeek endpoint) — it checks each abstract's text against the review's current
criteria and records every reason for exclusion, since a real PRISMA-style review has to account for
*why* each excluded abstract was excluded, not just that it was.

In [2]:
CRITERIA = {
    "criteria_version": 1,
    "required_study_type": "randomized controlled trial",
    "required_indication": "oncology",
    "min_year": 2015,
}


def screen_abstract(abstract: dict, criteria: dict) -> dict:
    text = abstract["text"].lower()
    reasons = []
    if criteria["required_study_type"] not in text:
        reasons.append("not_a_randomized_controlled_trial")
    if criteria["required_indication"] not in text:
        reasons.append("wrong_indication")
    if abstract["year"] < criteria["min_year"]:
        reasons.append("published_before_cutoff_year")
    verdict = "include" if not reasons else "exclude"
    return {
        "abstract_id": abstract["abstract_id"],
        "verdict": verdict,
        "exclusion_reasons": reasons,
        "criteria_version": criteria["criteria_version"],
    }


screening_results = [screen_abstract(a, CRITERIA) for a in corpus]

included_ids = {r["abstract_id"] for r in screening_results if r["verdict"] == "include"}
excluded = [r for r in screening_results if r["verdict"] == "exclude"]

print(f"Screened: {len(screening_results)} abstracts")
print(f"Included: {len(included_ids)}")
print(f"Excluded: {len(excluded)}")


Screened: 20 abstracts
Included: 5
Excluded: 15


## 3. The PRISMA-style funnel: accounting for every excluded abstract

A real systematic review has to report not just how many abstracts were excluded, but the reason
breakdown — this is exactly the "account for the full funnel" requirement `00-README.md` describes.

In [3]:
from collections import Counter

reason_counts = Counter()
for r in excluded:
    for reason in r["exclusion_reasons"]:
        reason_counts[reason] += 1

print("PRISMA-style screening funnel:")
print(f"  Total screened:      {len(screening_results)}")
print(f"  Included:            {len(included_ids)}")
print(f"  Excluded:            {len(excluded)}")
print("  Exclusion reasons (an abstract can have more than one):")
for reason, count in reason_counts.most_common():
    print(f"    - {reason}: {count}")

# Invariant worth checking on every real run: every abstract accounted for exactly once
assert len(included_ids) + len(excluded) == len(corpus)
print("\nOK: every abstract in the corpus is accounted for as exactly one of included/excluded.")


PRISMA-style screening funnel:
  Total screened:      20
  Included:            5
  Excluded:            15
  Exclusion reasons (an abstract can have more than one):
    - not_a_randomized_controlled_trial: 8
    - wrong_indication: 8
    - published_before_cutoff_year: 6

OK: every abstract in the corpus is accounted for as exactly one of included/excluded.


## 4. Extraction: structured data from included studies only

Extraction only runs against the screening stage's included set — never the full corpus — mirroring
Chapter 4's pipeline-gating design (the extraction stage is gated on screening's output).

In [4]:
def extract_structured_data(abstract: dict) -> dict:
    sample_match = re.search(r"enrolling (\d+) patients", abstract["text"])
    sample_size = int(sample_match.group(1)) if sample_match else None
    return {
        "abstract_id": abstract["abstract_id"],
        "sample_size": sample_size,
        "year": abstract["year"],
    }


included_abstracts = [a for a in corpus if a["abstract_id"] in included_ids]
extracted = [extract_structured_data(a) for a in included_abstracts]

print(f"Extracted structured data from {len(extracted)} included studies:")
for e in extracted:
    print(" ", e)

assert all(e["sample_size"] is not None for e in extracted)
print("\nOK: every included study yielded a parsed sample size -- nothing silently skipped extraction.")


Extracted structured data from 5 included studies:
  {'abstract_id': 'A001', 'sample_size': 280, 'year': 2023}
  {'abstract_id': 'A005', 'sample_size': 46, 'year': 2017}
  {'abstract_id': 'A014', 'sample_size': 510, 'year': 2023}
  {'abstract_id': 'A016', 'sample_size': 452, 'year': 2017}
  {'abstract_id': 'A019', 'sample_size': 22, 'year': 2017}

OK: every included study yielded a parsed sample size -- nothing silently skipped extraction.


## 5. Synthesis: aggregate the extracted evidence base

A minimal draft synthesis — real synthesis would be a narrative summary, but the aggregate numbers
here demonstrate the same shape: pulling one coherent picture out of the individually extracted
records, ready to route to a human medical-writer sign-off queue before becoming part of a report.

In [5]:
def synthesize(extracted: list) -> dict:
    total_n = sum(e["sample_size"] for e in extracted if e["sample_size"])
    years = [e["year"] for e in extracted]
    return {
        "num_included_studies": len(extracted),
        "total_patients_across_included_studies": total_n,
        "year_range": (min(years), max(years)) if years else None,
    }


synthesis = synthesize(extracted)
print("Draft synthesis (pre-human-review):")
for k, v in synthesis.items():
    print(f"  {k}: {v}")

assert synthesis["num_included_studies"] == len(included_ids)
assert synthesis["total_patients_across_included_studies"] == sum(e["sample_size"] for e in extracted)
print("\nOK: synthesis numbers reconcile exactly against the extraction stage's output -- this "
      "draft is what would route to a medical-writer sign-off queue next, never published directly.")


Draft synthesis (pre-human-review):
  num_included_studies: 5
  total_patients_across_included_studies: 1310
  year_range: (2017, 2023)

OK: synthesis numbers reconcile exactly against the extraction stage's output -- this draft is what would route to a medical-writer sign-off queue next, never published directly.


## 6. Tying it back

- The **screening funnel** (Section 3) demonstrates the PRISMA-style accounting discipline this
  domain requires: every abstract is either included or excluded, and every exclusion carries a
  recorded reason — not just a raw pass/fail count.
- **Extraction runs only against the included set** (Section 4), matching the multi-stage,
  gated-on-the-prior-stage pipeline shape Chapter 4 describes for Step Functions orchestration.
- **Synthesis aggregates exactly what extraction produced** (Section 5), with numbers that reconcile
  against the prior stage's output — the kind of internal consistency check worth running on every
  real batch, not just trusting the pipeline blindly.
- What's missing from this simplified demo, intentionally, because they're each their own chapter's
  concern: the `model_version`/`criteria_version` tagging on every screening decision (Chapter 7,
  notebook 04) and the human sign-off gate before anything becomes a final report artifact
  (Chapter 6, Chapter 8).